# Geodesic Walk — K=150 confirmation run + constant-step control

**Direction:** `research/directions/geodesic-walk.md` — ⟳ CONFIRMATION RUN (2026-06-24).

**What the K=30 pass left open.** The fractional on-manifold geodesic (`h + step_frac·(target−h)` + refit-local-tangent + project) *crawled* toward the target readout (RMSE 1.71→1.13 over 30 iters) and never reached it, with a decaying step size. Consistent with a **curvature barrier (true plateau)** — but K=30 can't rule out **slow-but-would-converge**, and the *fractional* nudge shrinks geometrically as `h` nears the target **even on a flat manifold**, so "step decayed" is not by itself evidence of a barrier.

**This run resolves both:**
1. **Run to large K.** `K_ITERS=150`, N=64, same checkpoint/data/probe, `STEP_FRAC=0.34`. Log full per-iter readout-RMSE + local-residual curves. **Decision rule (tail = last 50 iters):** if |Δ RMSE| over last 50 < 0.02 AND final RMSE > 0.5 → **PLATEAU/BARRIER**; if still descending materially toward ~0 → **slow convergence**.
2. **Constant-step control (the methodological fix).** Second variant: fixed `‖Δh‖` per iteration (set to the *first* fractional iteration's step norm), direction = normalized `(inject_state(h,target) − h)`, same local-tangent reprojection, K=150. If RMSE *still* plateaus far from 0 under a constant step while local-resid stays ≈ real → barrier is **real curvature**, not a schedule artifact. If constant-step reaches target → the K=30 barrier was just the decaying schedule. Both curves on the same axes.

Plus: regenerate obs-space waterfalls at the **final** iterate of each geodesic variant (object reach target? ghost gone?).

Self-contained cold-start; mirrors `geodesic_walk.ipynb`. Both printed tables (agent) and figures+PNGs to `/tmp/geodesic_k150/` (Sevan).

---
## 1 — Setup: model, probe, global + local-bank subspaces, warm-up to edit

In [ ]:
import sys, os
sys.path.insert(0, "../../..")   # repo root -> import pim
sys.path.insert(0, "../..")      # notebooks/ -> helpers

from dataclasses import replace
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display

import pim.eval as eval
from pim.extractors import LinearExtractor, StateDefinition, identity_mse, hungarian_mse
from pim.editors import (
    probe_decomposition, inject_state,
    fit_state_subspace, project_to_subspace, offmanifold_residual,
    fit_local_subspace, manifold_steer, manifold_steer_local,
)
from pim.eval.controllability import _rollout
from pim.world_models import load_checkpoint, load_dataset, make_test_loader

torch.manual_seed(0); np.random.seed(0)

CHECKPOINT_PATH = "../../../runs/gru/3_dset3_gru_persistentids_inview_400epochs/best_model.pt"
DATA_DIR        = "../../../datasets/4_fixed_refl_inview"
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE      = 512
NUM_WORKERS     = 6

N_OBJ           = 2
USE_HUNGARIAN   = False     # fixed reflectivities -> identity matching
SUBSPACE_VAR    = 0.90      # variance kept by the GLOBAL state-manifold PCA
LOCAL_K         = 512       # nearest neighbours per local tangent patch
LOCAL_VAR       = 0.90      # variance kept WITHIN a local patch
LOCAL_BANK_SIZE = 50_000    # visited-state bank subsample for the kNN

# --- experiment scale (CONFIRMATION RUN: large K) ---
N_GEO           = 64        # edit samples walked
N_ROLLOUT       = 15        # post-edit rollout length
K_ITERS         = 150       # geodesic-walk iterations  <-- bumped from 30
POCS_ITERS      = 50        # edit<->project alternations for the one-shot baselines

os.makedirs("/tmp/geodesic_k150", exist_ok=True)

model, ckpt_info = load_checkpoint(CHECKPOINT_PATH, device=DEVICE)
bundle = load_dataset(DATA_DIR, n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
test_loader = make_test_loader(test, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

# Teacher-force the test set -> bank of visited hidden states.
preds_tf, states_tf = eval.teacher_force(model, test_loader, device=DEVICE)

# Linear position probe.
state_def = StateDefinition(name="positions", state_shape=(N_OBJ, 2),
                            extract_fn=lambda b: b["positions"])
env_states_tf = test.positions[:, :-1, :N_OBJ, :]
vis_mask_tf   = test.is_visible[:, :-1, :N_OBJ].all(axis=2)
loss_fn       = hungarian_mse if USE_HUNGARIAN else identity_mse

linear = LinearExtractor(model.hidden_size, state_def, use_lstsq=True)
train_mse = linear.fit(states_tf, env_states_tf, mask=vis_mask_tf, loss_fn=loss_fn, device=DEVICE)
linear = linear.to(DEVICE).eval()

print(f"Model : {ckpt_info.run_name} (epoch {ckpt_info.epoch}, val_loss={ckpt_info.val_loss:.5f})")
print(f"Hidden: {model.hidden_size}  states_tf={states_tf.shape}")
print(f"Probe : linear position, train MSE={train_mse:.6f}  device={DEVICE}")

In [ ]:
# Global manifold subspace (on device) + on-device bank for local tangent fits.
subspace = fit_state_subspace(states_tf, var_threshold=SUBSPACE_VAR)
subspace_dev = replace(subspace,
    mean=subspace.mean.to(DEVICE), basis=subspace.basis.to(DEVICE),
    explained_variance_ratio=subspace.explained_variance_ratio.to(DEVICE))

_bank_all = states_tf.reshape(-1, model.hidden_size)
_sub = np.random.RandomState(0).choice(
    _bank_all.shape[0], size=min(LOCAL_BANK_SIZE, _bank_all.shape[0]), replace=False)
bank_dev = torch.from_numpy(_bank_all[_sub]).float().to(DEVICE)

# Off-manifold residual scale of REAL states (the on-manifold reference, global + local).
real_res_global = float(offmanifold_residual(
    torch.from_numpy(_bank_all[:5000]).float().to(DEVICE), subspace_dev).mean())
def _local_resid(h, n_probe=100):
    """Per-sample LOCAL-tangent off-manifold residual — the honest, curvature-aware detector."""
    res = []
    for i in range(min(n_probe, h.shape[0])):
        sub = fit_local_subspace(bank_dev, h[i], k_neighbors=LOCAL_K,
                                 var_threshold=LOCAL_VAR, bank_size=LOCAL_BANK_SIZE)
        res.append(float(offmanifold_residual(h[i:i+1], sub).mean()))
    return float(np.mean(res))
real_res_local = _local_resid(torch.from_numpy(_bank_all[_sub[:200]]).float().to(DEVICE))
print(f"global subspace: kept {subspace.n_components}/{subspace.hidden_size} "
      f"({subspace.total_explained:.4f} var)")
print(f"REAL-state off-manifold residual:  global={real_res_global:.4f}  local={real_res_local:.4f}")

# Warm up to the edit frame -> base hidden states we will edit/walk from.
N = min(N_GEO, edits.n_samples)
warm = eval.warm_up_to_edit(model, edits.obs[:N], edits.edit_frame,
                            n_viz=N, n_ctx_show=8, device=DEVICE)
h_base = warm.h_at_edit[:N]                                   # (N, H) cold-start point

# Target readout = post-edit (teleported) GT positions at the edit frame.
targets = edits.positions[:N, edits.edit_frame, :N_OBJ, :].reshape(N, N_OBJ * 2)

# Probe decomposition (on device).
A, b, A_pinv = probe_decomposition(linear)
h0  = torch.from_numpy(h_base).float().to(DEVICE)
tgt = torch.from_numpy(targets).float().to(DEVICE)
edit_fn = lambda h, t: inject_state(h, t, A, A_pinv, b)

def readout(h):                          # (.,H) -> (.,N_OBJ*2)
    return h @ A.T + b
def readout_rmse(h, t=tgt):
    return float((readout(h) - t).pow(2).mean().sqrt())
def resid_global(h):
    return float(offmanifold_residual(h, subspace_dev).mean())

@torch.no_grad()
def rollout_from_flat(h_array, n_rollout):
    """Roll out from each flat state; step 0 = decode (no advance)."""
    obs_all, h_all = [], []
    for i in range(h_array.shape[0]):
        h = torch.as_tensor(h_array[i], dtype=torch.float32, device=DEVICE).unsqueeze(0)
        o, hs = _rollout(model, h, n_rollout)
        obs_all.append(o); h_all.append(hs)
    return np.stack(obs_all), np.stack(h_all)

@torch.no_grad()
def decode_pos(h_array):
    t = torch.as_tensor(h_array, dtype=torch.float32, device=DEVICE)
    return linear(t).cpu().numpy()       # (...,N_OBJ,2)

print(f"\nh_base={h_base.shape}  edit_frame={edits.edit_frame}  N={N}  K_ITERS={K_ITERS}")
print(f"cold-start readout RMSE (unsteered vs target): {readout_rmse(h0):.4f}")

---
## 2 — One-shot baselines (reproduce the editability table)

The three one-shot editors as reference bars: **pseudoinv** (`inject_state`, off-manifold OK), **manifold** (`manifold_steer` against global PCA, POCS), **one-shot local** (`manifold_steer_local`: ONE local tangent fit at cold-start, no refit). High manifold/local readout RMSE = target unreachable in a fixed neighbourhood (the curvature barrier).

In [ ]:
h_pinv     = inject_state(h0, tgt, A, A_pinv, b)                                   # off-manifold
h_manifold = manifold_steer(h0, tgt, edit_fn, subspace_dev, n_iters=POCS_ITERS)    # global manifold
h_local    = manifold_steer_local(h0, tgt, edit_fn, bank_dev,                      # one-shot local
                                  k_neighbors=LOCAL_K, n_iters=POCS_ITERS,
                                  var_threshold=LOCAL_VAR, bank_size=LOCAL_BANK_SIZE)

print(f"{'edit':14s} {'readout RMSE':>13s} {'global resid':>13s} {'local resid':>12s}")
baseline_states = {"unsteered": h0, "pseudoinv": h_pinv,
                   "manifold": h_manifold, "one-shot local": h_local}
baseline_table = {}
for name, h in baseline_states.items():
    r_rmse, r_g, r_l = readout_rmse(h), resid_global(h), _local_resid(h)
    baseline_table[name] = (r_rmse, r_g, r_l)
    print(f"{name:14s} {r_rmse:13.4f} {r_g:13.4f} {r_l:12.4f}")
print(f"{'real states':14s} {'—':>13s} {real_res_global:13.4f} {real_res_local:12.4f}")

---
## 3 — The geodesic walk (two variants), run to K=150

At each iteration, starting from `h_at_edit`:
1. **Step toward target**, then **refit a FRESH local tangent subspace** around the new point and **project** onto it (re-localized every iter so the walk follows the manifold's curvature).
2. **Log** readout RMSE, local off-manifold residual, step size `‖Δh‖`.

Two step rules:
- **fractional** (`mode="fractional"`): `h_step = h + STEP_FRAC·(inject_state(h,target) − h)`. The original rule; its step shrinks geometrically as `h` nears the target *even on a flat manifold*.
- **constant** (`mode="constant"`): `h_step = h + c·d̂`, where `d̂` = normalized `(inject_state(h,target) − h)` and `c` = a **fixed** norm set to the *first* fractional iteration's step. Decoupling the step schedule from the geometry: if RMSE still plateaus far from 0 here while staying on-manifold, the barrier is real curvature, not a decaying schedule.

In [ ]:
from tqdm.auto import tqdm

STEP_FRAC = 0.34      # fraction of the full pseudoinverse jump per iteration (fractional mode)

@torch.no_grad()
def geodesic_walk(h_start, target, *, mode="fractional", k_iters=K_ITERS,
                  step_frac=STEP_FRAC, const_step=None,
                  k_neighbors=LOCAL_K, var_threshold=LOCAL_VAR,
                  bank=bank_dev, bank_size=LOCAL_BANK_SIZE, log_residual=True,
                  desc="geodesic walk"):
    """Iterated step-toward-target + refit-local-tangent + project, per sample.

    mode="fractional": h + step_frac*(inject_state(h,t) - h)  (geometric decay)
    mode="constant"  : h + const_step * unit(inject_state(h,t) - h)  (fixed ||Δstep||)

    Returns:
      h_final : (N, H) walked states
      log     : dict of arrays — 'rmse','resid_local' (N, k_iters+1); 'step' (N, k_iters).
    """
    assert mode in ("fractional", "constant")
    if mode == "constant":
        assert const_step is not None, "constant mode needs const_step"
    Nn = h_start.shape[0]
    h_out = torch.empty_like(h_start)
    rmse_log  = np.zeros((Nn, k_iters + 1))
    resid_log = np.full((Nn, k_iters + 1), np.nan)
    step_log  = np.zeros((Nn, k_iters))
    for i in tqdm(range(Nn), desc=desc, leave=False):
        h = h_start[i:i+1]
        t = target[i:i+1]
        rmse_log[i, 0] = float((readout(h) - t).pow(2).mean().sqrt())
        if log_residual:
            sub0 = fit_local_subspace(bank, h[0], k_neighbors=k_neighbors,
                                      var_threshold=var_threshold, bank_size=bank_size)
            resid_log[i, 0] = float(offmanifold_residual(h, sub0).mean())
        for k in range(k_iters):
            h_full = inject_state(h, t, A, A_pinv, b)        # full jump onto readout constraint
            d = h_full - h                                   # raw direction toward target
            if mode == "fractional":
                h_step = h + step_frac * d
            else:  # constant
                nrm = d.norm()
                dhat = d / nrm if float(nrm) > 1e-12 else d
                h_step = h + const_step * dhat
            sub = fit_local_subspace(bank, h_step[0], k_neighbors=k_neighbors,   # FRESH local patch
                                     var_threshold=var_threshold, bank_size=bank_size)
            h_proj = project_to_subspace(h_step, sub)        # project onto re-localized tangent
            step_log[i, k] = float((h_proj - h).norm())
            h = h_proj
            rmse_log[i, k + 1] = float((readout(h) - t).pow(2).mean().sqrt())
            if log_residual:
                resid_log[i, k + 1] = float(offmanifold_residual(h, sub).mean())
        h_out[i] = h[0]
    return h_out, {"rmse": rmse_log, "resid_local": resid_log, "step": step_log}

# --- Variant A: FRACTIONAL (the original rule), K=150 ---
h_geo_frac, log_frac = geodesic_walk(h0, tgt, mode="fractional", desc="fractional K=150")

# Constant step magnitude = the FIRST fractional iteration's mean step norm.
CONST_STEP = float(log_frac["step"][:, 0].mean())
print(f"first fractional-iteration mean step norm -> CONST_STEP = {CONST_STEP:.4f}")

# --- Variant B: CONSTANT step, K=150 ---
h_geo_const, log_const = geodesic_walk(h0, tgt, mode="constant", const_step=CONST_STEP,
                                       desc="constant K=150")

# Per-mode summary tables.
def summarize(log, label):
    rmse_m  = log["rmse"].mean(0); resid_m = log["resid_local"].mean(0); step_m = log["step"].mean(0)
    print(f"\n=== {label}  (STEP_FRAC={STEP_FRAC}, K={K_ITERS}, N={N}) ===")
    print(f"{'iter':>5} {'readout RMSE':>13} {'local resid':>12} {'step ||dh||':>12}")
    its = [k for k in [0,1,2,5,10,25,50,75,100,125,K_ITERS] if k <= K_ITERS]
    for k in its:
        s = f"{step_m[k-1]:12.4f}" if k >= 1 else f"{'—':>12}"
        print(f"{k:>5} {rmse_m[k]:13.4f} {resid_m[k]:12.4f} {s}")
    print(f"final readout RMSE = {rmse_m[-1]:.4f}  (cold start = {rmse_m[0]:.4f})")
    return rmse_m, resid_m, step_m

rmse_f, resid_f, step_f = summarize(log_frac,  "FRACTIONAL")
rmse_c, resid_c, step_c = summarize(log_const, "CONSTANT")
print(f"\nreal-state local resid reference = {real_res_local:.4f}")

---
## 4 — Tail-slope decision rule + convergence curves (both variants, same axes)

**Decision rule (per variant), tail = last 50 iters:** fit a line to the mean readout-RMSE over the last 50 iterations.
- |Δ RMSE across last 50| < 0.02 **AND** final RMSE > 0.5 → **PLATEAU / BARRIER**.
- still descending materially (and extrapolates toward ~0) → **slow convergence**.

If the **constant-step** variant *also* plateaus far from 0 while its local residual stays ≈ the real-state reference, the barrier is real curvature — not the fractional schedule's geometric decay.

In [ ]:
TAIL = 50

def tail_decision(rmse_m, label):
    """Fit a line to mean RMSE over the last TAIL iters; apply the brief's decision rule."""
    n = len(rmse_m)
    t0 = max(0, n - TAIL)
    x = np.arange(t0, n)
    y = rmse_m[t0:]
    slope, intercept = np.polyfit(x, y, 1)          # RMSE per iteration
    delta = slope * (len(x) - 1)                     # total Δ RMSE across the tail window
    final = rmse_m[-1]
    # iters-to-zero extrapolation (only meaningful if descending)
    iters_to_zero = (-final / slope) if slope < -1e-9 else np.inf
    if abs(delta) < 0.02 and final > 0.5:
        verdict = "PLATEAU / BARRIER"
    elif slope < -1e-3 and (final < 0.5 or iters_to_zero < 5000):
        verdict = "SLOW CONVERGENCE"
    else:
        verdict = "AMBIGUOUS"
    print(f"--- {label} tail decision (last {len(x)} iters) ---")
    print(f"  slope = {slope:+.5f} RMSE/iter   Δ over tail = {delta:+.4f}   final RMSE = {final:.4f}")
    print(f"  extrap iters-to-RMSE=0 = {iters_to_zero:.0f}" if np.isfinite(iters_to_zero)
          else "  extrap iters-to-RMSE=0 = inf (flat/ascending)")
    print(f"  VERDICT: {verdict}\n")
    return dict(slope=slope, delta=delta, final=final, iters_to_zero=iters_to_zero, verdict=verdict)

dec_frac  = tail_decision(rmse_f, "FRACTIONAL")
dec_const = tail_decision(rmse_c, "CONSTANT")

iters = np.arange(K_ITERS + 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

# (a) readout RMSE vs iter — both variants + baselines + tail-fit lines.
ax = axes[0]
ax.plot(iters, rmse_f, color="#0072B2", lw=2.5, marker="o", ms=2.5, label="fractional (mean)")
ax.plot(iters, rmse_c, color="#E69F00", lw=2.5, marker="s", ms=2.5, label="constant (mean)")
for d, c in [(dec_frac, "#0072B2"), (dec_const, "#E69F00")]:
    xt = np.arange(K_ITERS + 1 - TAIL, K_ITERS + 1)
    ax.plot(xt, d["slope"] * xt + (rmse_f if c == "#0072B2" else rmse_c)[-1] - d["slope"] * xt[-1],
            color=c, ls=":", lw=1.2, alpha=0.8)
for name, c, ls in [("pseudoinv", "#D55E00", ":"), ("manifold", "#009E73", "--"),
                    ("one-shot local", "#CC79A7", "-.")]:
    ax.axhline(baseline_table[name][0], color=c, ls=ls, lw=1.2, alpha=0.7, label=f"{name} (1-shot)")
ax.axhline(0.5, color="0.6", lw=0.8, ls="--", alpha=0.6)
ax.set_xlabel("iteration"); ax.set_ylabel("readout RMSE (decoded)")
ax.set_title("(a) convergence: RMSE -> 0 or plateau? (K=150)")
ax.grid(alpha=0.3); ax.legend(fontsize=7)

# (b) local off-manifold residual vs iter — both variants stay on-manifold?
ax = axes[1]
ax.plot(iters, resid_f, color="#0072B2", lw=2.2, marker="o", ms=2.5, label="fractional local resid")
ax.plot(iters, resid_c, color="#E69F00", lw=2.2, marker="s", ms=2.5, label="constant local resid")
ax.axhline(real_res_local, color="0.4", ls="--", lw=1.5, label=f"real-state local resid={real_res_local:.3f}")
ax.set_xlabel("iteration"); ax.set_ylabel("local off-manifold residual")
ax.set_title("(b) on-manifold? (residual vs real reference)")
ax.grid(alpha=0.3); ax.legend(fontsize=8)

# (c) step size vs iter — fractional decays geometrically; constant is fixed (modulo projection).
ax = axes[2]
ax.plot(np.arange(1, K_ITERS + 1), step_f, color="#0072B2", lw=2.0, marker="o", ms=2.5, label="fractional")
ax.plot(np.arange(1, K_ITERS + 1), step_c, color="#E69F00", lw=2.0, marker="s", ms=2.5, label="constant")
ax.axhline(CONST_STEP, color="0.5", ls=":", lw=1, label=f"CONST_STEP={CONST_STEP:.3f}")
ax.set_xlabel("iteration"); ax.set_ylabel("mean step size ||dh||")
ax.set_title("(c) step size per iteration"); ax.grid(alpha=0.3); ax.legend(fontsize=8)

fig.suptitle("Geodesic walk K=150: fractional vs constant-step", y=1.02, fontsize=13)
fig.tight_layout()
fig.savefig("/tmp/geodesic_k150/1_convergence.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)
print("saved /tmp/geodesic_k150/1_convergence.png")

In [ ]:
# Combined readout-space table: both geodesic finals vs one-shot baselines.
geo_final = {
    "GEODESIC fractional": h_geo_frac,
    "GEODESIC constant":   h_geo_const,
}
print("=== READOUT-SPACE COMPARISON (K=150 geodesics vs one-shot baselines) ===")
print(f"{'edit':20s} {'readout RMSE':>13s} {'global resid':>13s} {'local resid':>12s}")
for name, (r, g, l) in baseline_table.items():
    print(f"{name:20s} {r:13.4f} {g:13.4f} {l:12.4f}")
geo_readout = {}
for name, h in geo_final.items():
    r, g, l = readout_rmse(h), resid_global(h), _local_resid(h)
    geo_readout[name] = (r, g, l)
    print(f"{name:20s} {r:13.4f} {g:13.4f} {l:12.4f}")
print(f"{'real states':20s} {'—':>13s} {real_res_global:13.4f} {real_res_local:12.4f}")
for name in geo_final:
    print(f"on-manifold check {name}: local resid {geo_readout[name][2]:.3f} vs real "
          f"{real_res_local:.3f} (ratio {geo_readout[name][2]/real_res_local:.2f})")

---
## 5 — Observation-space outcome at the FINAL iterate (both geodesic variants)

A converged readout RMSE does NOT prove the model's generated observation moved. We roll out each edited state and ask in obs space: does the object reach the target (RMS vs the GT target render) and is the ghost/phantom (residual intensity at the pre-edit location) gone? Anchored at the **final** K=150 iterate of each variant.

In [ ]:
# Roll out every variant from its edited state (step 0 = decode, no advance).
variant_h = {
    "unsteered":          h0,
    "pseudoinv":          h_pinv,
    "manifold":           h_manifold,
    "one-shot local":     h_local,
    "geodesic frac":      h_geo_frac,
    "geodesic const":     h_geo_const,
}
roll_obs, roll_hs = {}, {}
for name, h in variant_h.items():
    o, hs = rollout_from_flat(h.detach().cpu().numpy(), N_ROLLOUT)
    roll_obs[name] = o; roll_hs[name] = hs
OBS_RES = roll_obs["unsteered"].shape[-1]
print("rollouts:", {k: v.shape for k, v in roll_obs.items()})

# Render the TARGET scene from post-edit GT positions (the obs a perfect edit should produce).
from pim.simulator.sim import Scene, SimConfig
from pim.simulator.renderer import render_scene
sim = test.config["dataset"]["sim"]
def make_cfg(n_frames):
    return SimConfig(seed=0, y_near=sim["y_near"], y_far=sim["y_far"], x_near=sim["x_near"],
                     x_far=sim["x_far"], n_objects=N_OBJ, radius=sim["radius"],
                     n_frames=n_frames, dt=sim["dt"], obs_res=sim["obs_res"],
                     refl_min=sim["refl_min"], refl_max=sim["refl_max"],
                     fixed_reflectivities=True, obs_noise_std=0.0, boundary="open",
                     always_in_frustum=False)
REFL = np.array([sim["refl_min"], sim["refl_max"]], dtype=np.float32)
RAD  = np.array([sim["radius"]] * N_OBJ, dtype=np.float32)
COLc = np.tile(np.array([[1, 1, 1]], dtype=np.float32), (N_OBJ, 1))

tgt_pos = edits.positions[:N, edits.edit_frame, :N_OBJ, :].astype(np.float32)
pre_pos = edits.positions[:N, edits.edit_frame - 1, :N_OBJ, :].astype(np.float32)
cfg1 = make_cfg(1)
tgt_render_id  = np.zeros((N, OBS_RES), dtype=np.int64)
tgt_render_int = np.zeros((N, OBS_RES), dtype=np.float32)
pre_render_id  = np.zeros((N, OBS_RES), dtype=np.int64)
for i in range(N):
    sc = Scene(positions=tgt_pos[i][None], velocities=np.zeros((1, N_OBJ, 2), np.float32),
               radii=RAD, colors=COLc, reflectivities=REFL, config=cfg1)
    _, rid, rint = render_scene(sc)
    tgt_render_id[i], tgt_render_int[i] = rid[0], rint[0]
    sc_pre = Scene(positions=pre_pos[i][None], velocities=np.zeros((1, N_OBJ, 2), np.float32),
                   radii=RAD, colors=COLc, reflectivities=REFL, config=cfg1)
    _, rid_pre, _ = render_scene(sc_pre)
    pre_render_id[i] = rid_pre[0]
print("target-render scenes built:", tgt_render_int.shape)
print("edit_object distribution:", np.bincount(edits.edit_object[:N]))

In [ ]:
# ---- Observation-space metrics ----
obs_u = roll_obs["unsteered"]
edit_obj = edits.edit_object[:N]

def rms(a, b):  return float(np.sqrt(((a - b) ** 2).mean()))
def dist_to_target_render(obs, step=0):  return rms(obs[:, step, :], tgt_render_int)
def obs_change(obs, step=0):             return rms(obs[:, step, :], obs_u[:, step, :])

unsteered_to_target = dist_to_target_render(obs_u)

# GHOST mask: rays hitting edited obj in PRE render but NOT in TARGET render.
ghost_mask = np.zeros((N, OBS_RES), dtype=bool)
for i in range(N):
    ghost_mask[i] = (pre_render_id[i] == edit_obj[i]) & (tgt_render_id[i] != edit_obj[i])
ghost_denom = ghost_mask.sum()
def ghost_ratio(obs, step=0):
    if ghost_denom == 0: return np.nan
    g_obs = obs[:, step, :][ghost_mask].mean()
    g_uns = obs_u[:, step, :][ghost_mask].mean()
    return float(g_obs / g_uns) if g_uns > 1e-6 else np.nan

print(f"ghost-region rays available: {int(ghost_denom)}  "
      f"(mean unsteered intensity there = {obs_u[:, 0, :][ghost_mask].mean():.3f})")
print(f"reference gap: unsteered scan vs target render (step0) = {unsteered_to_target:.4f}\n")

print("=== OBSERVATION-SPACE TABLE (step 0 = direct edit, final iterate) ===")
print(f"{'variant':16s} {'->target render':>15s} {'obs change':>11s} {'ghost ratio':>12s} {'%gap closed':>11s}")
obs_table = {}
for name, obs in roll_obs.items():
    d_t, d_c, g = dist_to_target_render(obs), obs_change(obs), ghost_ratio(obs)
    frac = 100 * (unsteered_to_target - d_t) / unsteered_to_target if unsteered_to_target > 1e-6 else 0.0
    obs_table[name] = (d_t, d_c, g)
    print(f"{name:16s} {d_t:15.4f} {d_c:11.4f} {g:12.3f} {frac:10.1f}%")
print("\nlower '->target render' = closer to a perfect edit; ghost ratio ~0 = removed, ~1 = phantom remains.")

# Step-resolved (persistence through rollout).
steps = np.arange(N_ROLLOUT)
to_tgt_step = {n: np.array([dist_to_target_render(o, s) for s in steps]) for n, o in roll_obs.items()}
chg_step    = {n: np.array([obs_change(o, s) for s in steps]) for n, o in roll_obs.items() if n != "unsteered"}
ghost_step  = {n: np.array([ghost_ratio(o, s) for s in steps]) for n, o in roll_obs.items()}

# Decoded-position distance to target (per object) — ties readout to obs.
tgt_pos_np = targets.reshape(N, N_OBJ, 2)
def dec_dist_per_obj(hs):
    pos0 = decode_pos(hs[:, 0])
    return np.sqrt(((pos0 - tgt_pos_np) ** 2).sum(-1)).mean(0)
print("\n=== DECODED-POSITION distance to target (step 0), per object ===")
print(f"{'variant':16s} " + " ".join(f"{'obj'+str(o):>9s}" for o in range(N_OBJ)) + f" {'mean':>9s}")
for name, hs in roll_hs.items():
    d = dec_dist_per_obj(hs)
    print(f"{name:16s} " + " ".join(f"{x:9.4f}" for x in d) + f" {d.mean():9.4f}")

In [ ]:
# Step-resolved obs-space figure.
COL = {"unsteered": "0.5", "pseudoinv": "#D55E00", "manifold": "#009E73",
       "one-shot local": "#CC79A7", "geodesic frac": "#0072B2", "geodesic const": "#E69F00"}
MK  = {"unsteered": None, "pseudoinv": "v", "manifold": "s", "one-shot local": "d",
       "geodesic frac": "o", "geodesic const": "^"}
def lw(n): return 2.4 if n.startswith("geodesic") else 1.3

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
ax = axes[0]
for n, v in to_tgt_step.items():
    ax.plot(steps, v, color=COL[n], marker=MK[n], ms=3, lw=lw(n), label=n)
ax.set_xlabel("rollout step"); ax.set_ylabel("RMS(generated obs, TARGET render)")
ax.set_title("(a) does the obs reach the target?"); ax.grid(alpha=0.3); ax.legend(fontsize=7)

ax = axes[1]
for n, v in chg_step.items():
    ax.plot(steps, v, color=COL[n], marker=MK[n], ms=3, lw=lw(n), label=n)
ax.set_xlabel("rollout step"); ax.set_ylabel("RMS obs change vs unsteered")
ax.set_title("(b) did the edit move the output?"); ax.grid(alpha=0.3); ax.legend(fontsize=7)

ax = axes[2]
for n, v in ghost_step.items():
    ax.plot(steps, v, color=COL[n], marker=MK[n], ms=3, lw=lw(n), label=n)
ax.axhline(1.0, color="0.7", ls="--", lw=1); ax.axhline(0.0, color="0.7", ls=":", lw=1)
ax.set_xlabel("rollout step"); ax.set_ylabel("ghost ratio (pre-edit location)")
ax.set_title("(c) is the phantom/ghost gone?"); ax.grid(alpha=0.3); ax.legend(fontsize=7)

fig.suptitle("Observation-space outcome (final iterate): geodesic variants vs baselines", y=1.02, fontsize=13)
fig.tight_layout()
fig.savefig("/tmp/geodesic_k150/2_obs_space_metrics.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)
print("saved /tmp/geodesic_k150/2_obs_space_metrics.png")

### 5b — Generated 1D scans + waterfalls at the FINAL iterate

Representative samples (largest GT teleport + resolvable ghost zone). 1D scan at the direct-edit step: unsteered / both geodesic finals / baselines vs the TARGET render (black dashed); pre-edit/ghost zone shaded. Then per-variant waterfalls with target-obj centroid (green) and pre-edit/ghost centroid (red).

In [ ]:
# Pick samples with the largest GT teleport AND a resolvable ghost zone.
teleport = np.linalg.norm(tgt_pos - pre_pos, axis=-1)[np.arange(N), edit_obj]
has_ghost = ghost_mask.sum(1) >= 3
SAMPLES = list(np.argsort(teleport * has_ghost)[::-1][:3])
print("representative samples:", SAMPLES, "teleport=", [round(float(teleport[s]), 2) for s in SAMPLES])

rays = np.arange(OBS_RES)
SCAN_STEP = 0
order = ["unsteered", "pseudoinv", "manifold", "one-shot local", "geodesic frac", "geodesic const"]

fig, axes = plt.subplots(len(SAMPLES), 1, figsize=(11, 3.0 * len(SAMPLES)), squeeze=False)
for r, smp in enumerate(SAMPLES):
    ax = axes[r][0]
    ax.plot(rays, tgt_render_int[smp], color="k", ls="--", lw=1.6, label="TARGET render", zorder=5)
    gz = np.where(ghost_mask[smp])[0]
    if gz.size:
        ax.axvspan(gz.min() - 0.5, gz.max() + 0.5, color="red", alpha=0.10, zorder=0, label="ghost zone")
    for n in order:
        ax.plot(rays, roll_obs[n][smp, SCAN_STEP], color=COL[n],
                lw=2.4 if n.startswith("geodesic") else 1.3,
                alpha=0.95 if n.startswith("geodesic") else 0.8,
                label=n, zorder=4 if n.startswith("geodesic") else 2)
    ax.set_title(f"sample {smp}  (edited obj {edit_obj[smp]}, teleport={teleport[smp]:.2f})")
    ax.set_xlabel("ray index"); ax.set_ylabel("intensity"); ax.set_ylim(-0.02, 1.05)
    ax.grid(alpha=0.25); ax.legend(fontsize=7, ncol=3, loc="upper right")
fig.suptitle(f"Generated 1D scans at direct-edit step (final iterate) — geodesics vs TARGET render", y=1.005, fontsize=12)
fig.tight_layout()
fig.savefig("/tmp/geodesic_k150/3_scans.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)
print("saved /tmp/geodesic_k150/3_scans.png")

In [ ]:
# Waterfalls: rows = samples, cols = variants. Green = target-obj centroid, red = pre-edit/ghost centroid.
def centroid(mask_row):
    idx = np.where(mask_row)[0]
    return idx.mean() if idx.size else np.nan

fig, axes = plt.subplots(len(SAMPLES), len(order),
                         figsize=(2.5 * len(order), 3.0 * len(SAMPLES)), squeeze=False)
for r, smp in enumerate(SAMPLES):
    tgt_cx = centroid(tgt_render_id[smp] == edit_obj[smp])
    pre_cx = centroid(pre_render_id[smp] == edit_obj[smp])
    for c, n in enumerate(order):
        ax = axes[r][c]
        ax.imshow(roll_obs[n][smp], aspect="auto", origin="upper", cmap="gray",
                  vmin=0, vmax=1, interpolation="nearest")
        if not np.isnan(tgt_cx): ax.axvline(tgt_cx, color="#00E676", ls="-", lw=1.4, alpha=0.9)
        if not np.isnan(pre_cx): ax.axvline(pre_cx, color="#FF5252", ls="--", lw=1.4, alpha=0.9)
        if r == 0: ax.set_title(n, fontsize=9)
        if c == 0: ax.set_ylabel(f"smp {smp}\nframe", fontsize=9)
        ax.set_xlabel("ray", fontsize=8)
axes[0][0].plot([], [], color="#00E676", lw=2, label="target loc")
axes[0][0].plot([], [], color="#FF5252", ls="--", lw=2, label="ghost loc")
axes[0][0].legend(loc="upper right", fontsize=6)
fig.suptitle("Generated waterfalls (final iterate): green = where edited obj SHOULD be, red = ghost zone\n"
             "(good edit: bright streak at green, nothing at red)", y=1.015, fontsize=12)
fig.tight_layout()
fig.savefig("/tmp/geodesic_k150/4_waterfalls.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)
print("saved /tmp/geodesic_k150/4_waterfalls.png")

---
## 6 — Verdict (auto-summarized)

Reads the computed numbers and prints the confirmation-run answers: plateau vs convergence per variant (tail-slope rule), and whether the constant-step control implies a real curvature barrier vs a schedule artifact. See `research/scratch/2026-06-24-geodesic-walk-k150.md`. **Not promoted to findings.**

In [ ]:
print("=" * 72)
print(f"GEODESIC WALK K=150 — AUTO VERDICT  (N={N}, K={K_ITERS}, STEP_FRAC={STEP_FRAC}, CONST_STEP={CONST_STEP:.3f})")
print("=" * 72)

print("\n(1) CONVERGENCE / PLATEAU (tail-slope rule, last 50 iters):")
for label, rmse_m, dec in [("fractional", rmse_f, dec_frac), ("constant", rmse_c, dec_const)]:
    print(f"  {label:11s}: cold={rmse_m[0]:.3f} -> final={rmse_m[-1]:.3f} | "
          f"tail slope={dec['slope']:+.5f}/iter, Δ={dec['delta']:+.4f} | {dec['verdict']}")

print("\n(2) ON-MANIFOLD (local resid vs real-state ref = {:.3f}):".format(real_res_local))
for name, (r, g, l) in geo_readout.items():
    print(f"  {name:20s}: local resid={l:.3f}  (ratio {l/real_res_local:.2f})")

print("\n(3) BARRIER REAL vs SCHEDULE ARTIFACT (constant-step control):")
cf = dec_const["final"]; ff = dec_frac["final"]
if dec_const["verdict"] == "PLATEAU / BARRIER" and geo_readout["GEODESIC constant"][2] < 2*real_res_local:
    print(f"  Constant-step ALSO plateaus far from 0 (final RMSE={cf:.3f}) while ON-manifold")
    print(f"  => barrier is REAL CURVATURE, not the fractional schedule's decay.")
elif dec_const["verdict"] == "SLOW CONVERGENCE" or cf < 0.5:
    print(f"  Constant-step REACHES (or descends toward) target (final RMSE={cf:.3f})")
    print(f"  => the K=30 'barrier' was largely the decaying fractional schedule.")
else:
    print(f"  Constant-step final RMSE={cf:.3f}, verdict={dec_const['verdict']} — see curves.")

print("\n(4) OBSERVATION SPACE at final iterate (step 0):")
print(f"  unsteered ->target render = {obs_table['unsteered'][0]:.3f}  (the gap a perfect edit closes)")
for n in ["geodesic frac", "geodesic const"]:
    d_t, _, g = obs_table[n]
    frac = 100 * (unsteered_to_target - d_t) / unsteered_to_target
    reached = "REACHES" if frac > 60 else ("PARTIAL" if frac > 15 else "does NOT reach")
    ghost_v = "ghost RESOLVED" if g < 0.3 else ("ghost PARTIAL" if g < 0.7 else "GHOST REMAINS")
    print(f"  {n:16s}: ->target={d_t:.3f} ({frac:.0f}% gap closed, {reached}); ghost ratio={g:.2f} ({ghost_v})")

print("\nPNGs: /tmp/geodesic_k150/{1_convergence,2_obs_space_metrics,3_scans,4_waterfalls}.png")